# J1 — Des textes à une recherche utile

**NLP Avancé (BERT, GPT, Hugging Face) · M2 IA · Jour 1 · 240 min · Chrys NIONGOLO**

Travail en binôme : alterner la personne qui conduit et celle qui explique. Enregistrer une copie dans Drive, puis conserver le notebook exécuté et le JSON final.

## À la fin, vous saurez

- Expliquer ce que représente un token et inspecter les cas limites français.
- Construire une recherche TF-IDF, puis la comparer à des embeddings.
- Mesurer Recall@k et MRR, et justifier une erreur avec un exemple.

## Parcours de l’après-midi

| Étape | Minutes |
|---|---:|
| Cadrage, données et tokenisation | 35 |
| TF-IDF et recherche | 50 |
| Tokenizer BPE et expériences | 40 |
| Embeddings et comparaison | 55 |
| Analyse d’erreurs, test et restitution | 60 |

Les durées sont des budgets d’activité incluant lecture, discussion, essais et débrief ; ce ne sont pas des durées de calcul promises. Les corpus intégrés sont originaux, fictifs et minuscules. Ils servent à comprendre un protocole, pas à certifier une qualité métier.

**Règle d’expérience :** écrire une prédiction avant de lancer une variante ; changer un seul facteur ; choisir sur la validation ; ouvrir le test après ce choix.


## 0. Préparer Colab

Importer ce fichier dans https://colab.research.google.com/ via Fichier → Importer.
Choisir Exécution → Modifier le type d’exécution → Python 3 → GPU T4 si disponible.
La disponibilité du T4 dépend des quotas Colab. Aucun compte Hugging Face n’est requis
pour les modèles publics utilisés ici. Une connexion internet est nécessaire pour les
bibliothèques et les poids ; les données pédagogiques sont déjà dans le notebook.

Exécuter la cellule suivante **avant les autres imports**. Si elle demande un redémarrage,
choisir Exécution → Redémarrer la session puis relancer depuis le début. Ne pas réinstaller
un autre PyTorch dans un runtime CUDA fonctionnel. Les versions sont volontairement fixées.
Aucun entraînement n’est lancé par cette cellule. Aucun envoi au Hub n’est automatique.

**Runpod L4 facultatif :** ce notebook fonctionne dans Jupyter avec Python 3.10+
et PyTorch CUDA préinstallé. Les chemins sont relatifs ; aucun import Google Colab
n’est requis. Arrêter le Pod après sauvegarde des résultats pour interrompre sa facturation.

**À commenter à l'oral.** Fixer les versions et la graine facilite la comparaison.
Cela ne garantit pas des sorties identiques sur tous les GPU ou tous les runtimes.
Redémarrer si la cellule d'installation le demande, puis reprendre depuis le début.
Pour cette correction, exécuter toutes les cellules dans l'ordre ; l'export JSON vient en dernier.


In [ ]:
PINNED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.6', 'scikit-learn': '1.7.2', 'transformers': '4.56.2', 'tokenizers': '0.22.0', 'huggingface-hub': '0.35.3', 'accelerate': '1.10.1', 'sentence-transformers': '5.1.1'}
import sys, subprocess, importlib.metadata as metadata
assert sys.version_info >= (3, 10), "Utiliser Python 3.10 ou ultérieur."
module_names = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",
                "huggingface-hub": "huggingface_hub", "rouge-score": "rouge_score"}
to_install = []
loaded_mismatch = []
for package, version in PINNED.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", version) != version:
        loaded_mismatch.append(package)
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        to_install.append(f"{package}=={version}")
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
if loaded_mismatch:
    raise RuntimeError("Redémarrer la session Colab, puis relancer depuis le début : "
                       + ", ".join(loaded_mismatch))
print("Versions de référence installées. Continuer avec la cellule suivante.")


In [ ]:
import os, json, time, random, platform, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
versions = {p: metadata.version(p) for p in PINNED}
versions["python"] = platform.python_version()
results = {"seed": SEED, "versions": versions, "validation": "exécuté par le binôme"}
OUTPUT = Path("resultats")
OUTPUT.mkdir(exist_ok=True)
display(pd.Series(versions, name="version effective"))


## **1. Partir d’une question concrète et examiner la tokenisation**

Notre service client possède douze fiches. Le système doit retrouver la bonne fiche,
sans générer une réponse. Avant le calcul, associer les trois premières requêtes à une fiche.

**Points clés pour l'oral :**
- La séparation par espace garde `l'appli` et `ré-expédition` comme des blocs ; elle sépare `15,50` de `€`.
- La regex précise affichée ci-dessous sépare `l'appli` en `l`, `'`, `appli`, et `15,50 €` en `15`, `,`, `50`, `€`.
- Ces choix produisent des unités de surface. Ils ne sont pas universellement bons ou mauvais : la tâche décide ce qu'il faut conserver ou reconstruire.

**Question avant exécution :** que risque-t-on en supprimant la négation dans « Je ne veux plus annuler » ?


In [ ]:
import re
faq = [
    ("suivi", "Suivre un colis", "Consultez le numéro de suivi pour localiser votre colis et connaître les étapes du transport."),
    ("non_recu", "Colis indiqué livré", "Le colis est marqué livré mais absent : vérifiez le lieu de dépôt puis contactez le support."),
    ("facture", "Télécharger une facture", "Retrouvez la facture et le justificatif comptable dans le détail de votre commande."),
    ("double", "Double débit", "Deux paiements identiques : vérifiez les autorisations bancaires puis signalez un éventuel double débit."),
    ("mot_passe", "Retrouver son accès", "Utilisez le lien de réinitialisation si votre mot de passe est oublié."),
    ("email", "Changer le courriel", "L'adresse email du compte peut être modifiée depuis les paramètres du profil."),
    ("retour", "Renvoyer un article", "Ouvrez une demande de retour et consultez les conditions avant de renvoyer le produit."),
    ("etiquette", "Bordereau de retour", "Téléchargez l'étiquette de retour depuis votre demande ; le support aide en cas d'échec."),
    ("bug", "Application qui plante", "Mettez à jour l'application puis signalez le plantage, votre appareil et les étapes de reproduction."),
    ("image", "Photos absentes", "Si les images des produits ne chargent pas, vérifiez la connexion puis actualisez le navigateur."),
    ("annuler", "Annuler une commande", "Demandez l'annulation au support avant l'expédition. Une commande expédiée suit la procédure de retour."),
    ("conserver", "Conserver une commande", "Pour conserver votre achat, ne demandez pas son annulation. Contactez le support si une demande est déjà en cours."),
]
docs = pd.DataFrame(faq, columns=["id", "titre", "texte"])
dev_queries = [
    ("Le colis n'avance plus dans le suivi", "suivi"),
    ("Il me faut une pièce pour ma comptabilité", "facture"),
    ("Ma banque affiche deux prélèvements", "double"),
    ("J'ai perdu mon secret de connexion", "mot_passe"),
    ("L'appli se ferme brutalement", "bug"),
    ("Je ne veux pas annuler mon achat", "conserver"),
    ("Le livreur dit livré mais je n'ai rien", "non_recu"),
    ("Comment imprimer le document de renvoi ?", "etiquette"),
]
# Ne pas utiliser cette liste pour ajuster les paramètres.
final_queries = [
    ("J'ai besoin du reçu de mon achat", "facture"),
    ("Les visuels sont tous invisibles", "image"),
    ("Je souhaite utiliser une autre adresse électronique", "email"),
    ("Je vais vous restituer cet article", "retour"),
    ("Je souhaite stopper ma commande avant son départ", "annuler"),
    ("J'aimerais garder cette commande finalement", "conserver"),
]
display(docs)
exemple = "Je ne veux plus annuler l'appli à 15,50 € : ré-expédition ?"
print("Espace :", exemple.split())
print("Expression régulière :", re.findall(r"\w+|[^\w\s]", exemple))


In [ ]:
reponses = {
    "negation": "Supprimer ne/pas/plus peut rapprocher des intentions opposées. Le TF-IDF utilisé ici ne retire pas les stop words : ne et pas sont conservés. Il ne représente toutefois pas explicitement la portée de la négation ; on vérifie le classement au lieu de présumer une erreur.",
    "unite_token": "split() garde l'appli et ré-expédition en un bloc et sépare 15,50 de €. La regex affichée produit un autre découpage. Un token n'est pas forcément un mot ; le choix se juge selon la tâche.",
    "hypothese_semantique": "Hypothèse à tester : un encodeur de phrases pourrait mieux retrouver certaines paraphrases que le recouvrement lexical. Une similarité élevée ne garantit ni la même intention ni une réponse correcte."
}
print(json.dumps(reponses, indent=2, ensure_ascii=False))


### **Repères formateur — éléments de correction**

La négation change l’action demandée. Supprimer « ne » et « pas » comme stop words peut rapprocher des intentions opposées. Un token est une unité du tokenizer : ni nécessairement un mot ni nécessairement une syllabe. Une bonne tokenisation ne garantit pas une bonne compréhension.


## **2. Une baseline classique : TF-IDF**

**Points clés pour l'oral :**
- **TF** compte les occurrences d'un terme dans un document. **IDF** réduit le poids relatif des termes présents dans beaucoup de documents.
- Ici, scikit-learn utilise une IDF lissée, puis normalise chaque vecteur en norme L2. Le cosinus final ne croît donc pas simplement de façon linéaire avec le nombre d'occurrences.
- **Fossé lexical :** `secret de connexion` et la fiche `mot de passe` partagent notamment `de`. Le score n'est donc pas forcément nul ; le système lexical ne relie pas à lui seul les deux expressions.
- Aucun stop word n'est supprimé dans cette configuration. Les termes d'au moins deux caractères sont retenus après passage en minuscules et normalisation des accents.

**Avant exécution :** quelle fiche « pièce pour ma comptabilité » va-t-elle retrouver ? Lire les termes et les voisins avant de conclure.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
NGRAM_RANGE = (1, 1)
vectorizer = TfidfVectorizer(ngram_range=NGRAM_RANGE, strip_accents="unicode")
document_vectors = vectorizer.fit_transform(docs.texte)
def lexical_scores(queries):
    return cosine_similarity(vectorizer.transform(queries), document_vectors)
terms = vectorizer.get_feature_names_out()
weights = document_vectors[2].toarray().ravel()
display(pd.DataFrame({"terme": terms, "poids": weights}).sort_values("poids", ascending=False).head(8))
query = "Il me faut une pièce pour ma comptabilité"
ranking = np.argsort(-lexical_scores([query])[0])[:3]
display(docs.iloc[ranking].assign(score=lexical_scores([query])[0][ranking]))
print("Dimensions documents × vocabulaire :", document_vectors.shape)


### **Mesurer une recherche**

Recall@3 vaut 1 si la bonne fiche figure dans les trois premiers résultats, sinon 0.
Le rang réciproque vaut 1/rang de la bonne fiche ; MRR en fait la moyenne.
Ici chaque question a une seule fiche attendue. Des données réelles peuvent accepter plusieurs réponses.

**Expérience A (15 min).** Comparer unigrammes et unigrammes+bigrams sur la validation.
Consigner le changement et garder les mêmes requêtes. Ne modifier ni les fiches ni les labels pour améliorer le score.


In [ ]:
# Définition de la fonction de calcul des métriques
def retrieval_metrics(scores, queries):
    expected = [docs.index[docs.id == expected_id][0] for _, expected_id in queries]
    order = np.argsort(-scores, axis=1, kind="stable")
    ranks = np.array([np.flatnonzero(order[i] == target)[0] + 1 for i, target in enumerate(expected)])
    return {"recall_at_1": float(np.mean(ranks <= 1)), "recall_at_3": float(np.mean(ranks <= 3)),
            "mrr": float(np.mean(1 / ranks)), "n": len(queries)}

# Deux expériences sur les mêmes requêtes, toutes deux conservées.
lexical_runs = {}
experiments = []
for ngram_range in [(1, 1), (1, 2)]:
    candidate = TfidfVectorizer(ngram_range=ngram_range, strip_accents="unicode")
    vectors = candidate.fit_transform(docs.texte)
    scores = cosine_similarity(candidate.transform([q for q, _ in dev_queries]), vectors)
    metrics = retrieval_metrics(scores, dev_queries)
    lexical_runs[ngram_range] = {"vectorizer": candidate, "vectors": vectors,
                                "scores": scores, "metrics": metrics}
    experiments.append({"method": "tfidf", "ngram_range": ngram_range, **metrics})

# Critère fixé : MRR de validation ; à égalité, conserver les unigrammes.
NGRAM_RANGE = max(lexical_runs, key=lambda n: lexical_runs[n]["metrics"]["mrr"])
selected = lexical_runs[NGRAM_RANGE]
vectorizer, document_vectors = selected["vectorizer"], selected["vectors"]
dev_scores_lexical, metrics_lexical = selected["scores"], selected["metrics"]
def lexical_scores(queries):
    return cosine_similarity(vectorizer.transform(queries), document_vectors)

display(pd.DataFrame(experiments))
print("Configuration TF-IDF retenue sur validation :", NGRAM_RANGE)
assert retrieval_metrics(np.eye(len(docs))[:2], [("q1", docs.id.iloc[0]), ("q2", docs.id.iloc[1])])["mrr"] == 1.0


## **Bonus formateur : BM25, une autre recherche lexicale**

Ce bonus remplace une variante de recherche ou dix minutes de restitution dans les 240 minutes prévues.
Le parcours étudiant reste réalisable sans ce bonus.

- **Saturation :** répéter un terme augmente sa contribution, mais avec un gain qui diminue.
- **Longueur :** à fréquence du terme identique, la longueur du document intervient relativement à la longueur moyenne. Le paramètre `b` règle cet effet ; `k1` règle la saturation.
- **Variante utilisée :** IDF positive $\ln(1 + (N-df+0,5)/(df+0,5))$, puis contribution $IDF\,f(k_1+1)/(f+k_1(1-b+bL/\bar L))$ par terme distinct de la requête. `N` compte les documents, `df` ceux contenant le terme, `f` ses occurrences, `L` les tokens du document et $\bar L$ leur moyenne.
- Les fiches sont indexées en unigrammes. Un résultat différent du TF-IDF unigrammes+bigrammes ne permet donc pas d'isoler le seul effet de BM25 ; comparer aussi au TF-IDF unigramme conservé dans le tableau.

Cette version ignore les répétitions d'un terme dans la requête. Un terme absent du vocabulaire n'ajoute rien au score.
Les paramètres sont fixés avant de lire le test. L'IDF positive est documentée par [Lucene](https://lucene.apache.org/core/9_9_1/core/org/apache/lucene/search/similarities/BM25Similarity.html).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

class BM25Search:
    def __init__(self, b=0.75, k1=1.5):
        if not 0 <= b <= 1 or k1 <= 0:
            raise ValueError("Utiliser 0 <= b <= 1 et k1 > 0.")
        self.b = b
        self.k1 = k1
        self.vectorizer = CountVectorizer(strip_accents="unicode")

    def fit(self, corpus):
        # Calcul des fréquences de termes (TF brut)
        tf = self.vectorizer.fit_transform(corpus).toarray()
        self.doc_len = tf.sum(axis=1)
        self.avg_doc_len = self.doc_len.mean()

        # Variante BM25 à IDF positive (cf. Lucene)
        N = len(corpus)
        # Nombre de documents contenant chaque terme
        df = (tf > 0).sum(axis=0)
        self.idf = np.log((N - df + 0.5) / (df + 0.5) + 1.0)
        self.tf = tf
        return self

    def transform(self, queries):
        query_tf = self.vectorizer.transform(queries).toarray()
        scores = []

        for q in query_tf:
            # Identifier les index des termes présents dans la requête
            query_indices = np.where(q > 0)[0]
            if len(query_indices) == 0:
                scores.append(np.zeros(self.tf.shape[0]))
                continue

            # Calcul de la formule BM25 pour chaque document
            doc_scores = np.zeros(self.tf.shape[0])
            for idx in query_indices:
                tf_term = self.tf[:, idx]
                idf_term = self.idf[idx]

                # Formule BM25
                numerator = tf_term * (self.k1 + 1)
                denominator = tf_term + self.k1 * (1 - self.b + self.b * (self.doc_len / self.avg_doc_len))
                doc_scores += idf_term * (numerator / denominator)
            scores.append(doc_scores)

        return np.array(scores).reshape(len(query_tf), self.tf.shape[0])

# Indexation des fiches et évaluation BM25 sur validation
bm25 = BM25Search().fit(docs.texte)
dev_scores_bm25 = bm25.transform([q for q, _ in dev_queries])
metrics_bm25 = retrieval_metrics(dev_scores_bm25, dev_queries)

print("Métriques de validation (BM25) :", metrics_bm25)
results["validation_bm25"] = metrics_bm25
experiments.append({"method": "bm25", "b": bm25.b, "k1": bm25.k1, **metrics_bm25})


### **Repères formateur — éléments de correction**

Les bigrammes peuvent préserver des expressions mais accroissent la rareté. Il n’existe pas de gagnant imposé. Un score ex aequo sur une matrice de zéros dépend ici de l’ordre stable des fiches : faire repérer ce cas. Le petit test d’identité vérifie le calcul de la métrique, pas la qualité du moteur.


## **3. Entraîner un tokenizer BPE (Byte-Pair Encoding)**

**Points clés pour l'oral :**
- BPE est un algorithme courant de sous-mots. Dans cet exercice, il apprend des fusions de paires à partir des caractères du corpus ; d'autres tokenizers utilisent d'autres unités ou algorithmes.
- Comparer des tailles cibles de **120** et **240** sur les mêmes textes. Davantage de fusions peuvent réduire certains découpages ; lire les sorties réelles au lieu d'imposer une segmentation attendue.
- À dimension d'embedding fixée, davantage d'entrées demanderaient une table de vecteurs plus grande. Ici, nous entraînons seulement un tokenizer, sans réseau de neurones.
- Le vocabulaire effectif peut être inférieur à la cible ; un caractère absent du corpus peut rester inconnu.

**Expérience :** comparer une négation, un nom inédit et une faute. Ce BPE miniature ne remplace pas le tokenizer d'un modèle préentraîné.


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

def train_bpe(vocab_size):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator(docs.texte.tolist(), trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=1, special_tokens=["[UNK]", "[PAD]"]))
    return tok

phrases = ["Je ne veux pas annuler", "réinitialisation du mot de passe", "N'Djamena", "coliiiiis 😅"]
bpe_rows = []
for size in [120, 240]:
    tok = train_bpe(size)
    for phrase in phrases:
        encoded = tok.encode(phrase)
        bpe_rows.append({"vocab": tok.get_vocab_size(), "phrase": phrase,
                         "tokens": encoded.tokens, "nombre": len(encoded.ids),
                         "inconnus": encoded.tokens.count("[UNK]")})

display(pd.DataFrame(bpe_rows))

### **Repères formateur — éléments de correction**

Ce BPE minuscule illustre les fusions, pas une couverture réaliste du français. Des caractères absents du corpus deviennent inconnus. Le vocabulaire réellement obtenu peut être inférieur à la cible. Un tokenizer byte-level limiterait ce problème de couverture, au prix d’autres arbitrages. Changer le tokenizer d’un modèle préentraîné sans réapprendre ses embeddings casse la correspondance identifiant-vecteur.


## **4. Passer aux embeddings denses de phrases**

**Points clés pour l'oral :**
- Ce modèle encode un texte en **384 dimensions**. TF-IDF possède une coordonnée par terme de son vocabulaire et produit surtout des zéros ; ce petit corpus n'exige pas un vecteur « géant ».
- Le modèle préentraîné vise à rapprocher des phrases similaires. Vérifier s'il retrouve ici `mot_passe` pour « secret de connexion » ; cette réussite n'est pas garantie pour toute paraphrase.
- Une proximité reste un signal, pas une preuve d'équivalence d'intention. Examiner notamment négations, références exactes et requêtes ambiguës.
- `encode` ne fine-tune aucun poids. Après normalisation L2, le produit scalaire des vecteurs correspond à leur cosinus.

Si les poids sont indisponibles, mettre `USE_PRETRAINED = False` avant d'exécuter la cellule.
La comparaison et le test restent utilisables avec TF-IDF et BM25 ; aucune mesure dense n'est inventée.


In [ ]:
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
USE_PRETRAINED = True
embedder = None
metrics_semantic = None

if USE_PRETRAINED:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = embedder.encode(docs.texte.tolist(), normalize_embeddings=True)
    def semantic_scores(queries):
        q = embedder.encode(queries, normalize_embeddings=True)
        return q @ embeddings.T
    dev_scores_semantic = semantic_scores([q for q, _ in dev_queries])
    metrics_semantic = retrieval_metrics(dev_scores_semantic, dev_queries)
    print("Dimensions :", embeddings.shape, "; métriques :", metrics_semantic)
    experiments.append({"method": "embedding", "model": EMBEDDING_MODEL, **metrics_semantic})
    assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5)
else:
    print("Embeddings préentraînés non exécutés.")


In [ ]:
analysis = []
for i, (query, expected) in enumerate(dev_queries):
    row = {"question": query, "attendu": expected,
           "tfidf": docs.id.iloc[int(dev_scores_lexical[i].argmax())],
           "bm25": docs.id.iloc[int(dev_scores_bm25[i].argmax())]}
    if embedder is not None:
        row["embedding"] = docs.id.iloc[int(dev_scores_semantic[i].argmax())]
    analysis.append(row)
display(pd.DataFrame(analysis))

if embedder is not None:
    from sklearn.decomposition import PCA
    # On encode également les requêtes de dev pour visualiser leur alignement avec les fiches
    dev_query_texts = [q for q, _ in dev_queries]
    query_embeddings = embedder.encode(dev_query_texts, normalize_embeddings=True)

    # PCA combinée pour garder le même espace de projection
    all_vectors = np.vstack([embeddings, query_embeddings])
    pca = PCA(n_components=2, random_state=SEED)
    all_xy = pca.fit_transform(all_vectors)

    print("Variance expliquée par les deux axes :", float(pca.explained_variance_ratio_.sum()))
    xy_docs = all_xy[:len(docs)]
    xy_queries = all_xy[len(docs):]

    plt.figure(figsize=(14, 8))

    # Tracer les documents (Fiches)
    plt.scatter(xy_docs[:, 0], xy_docs[:, 1], color='#1f77b4', s=150, label='Fiches de Référence (FAQ)', marker='o', edgecolors='black', alpha=0.8)
    # Tracer les requêtes utilisateur (Queries)
    plt.scatter(xy_queries[:, 0], xy_queries[:, 1], color='#ff7f0e', s=120, label='Requêtes de Validation', marker='^', edgecolors='black', alpha=0.9)

    # Annoter les documents
    for i, name in enumerate(docs.id):
        plt.annotate(
            name, (xy_docs[i, 0], xy_docs[i, 1]),
            textcoords="offset points", xytext=(0,10), ha='center',
            fontsize=10, fontweight='bold', color='#1f77b4',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#1f77b4", lw=1, alpha=0.7)
        )

    # Annoter les requêtes (en plus petit et italique)
    for i, text in enumerate(dev_query_texts):
        # Raccourcir le texte de la requête pour ne pas surcharger la figure
        short_text = text[:30] + '...' if len(text) > 30 else text
        plt.annotate(
            short_text, (xy_queries[i, 0], xy_queries[i, 1]),
            textcoords="offset points", xytext=(0,-15), ha='center',
            fontsize=8, style='italic', color='#d62728',
            bbox=dict(boxstyle="sawtooth,pad=0.2", fc="#fff2e6", ec="#ff7f0e", lw=0.8, alpha=0.8)
        )

    plt.title("PCA 2D : vue partielle des fiches et requêtes de validation", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Composante Principale 1", fontsize=12)
    plt.ylabel("Composante Principale 2", fontsize=12)
    plt.legend(loc='upper left', fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


## **Comparer les méthodes sur la validation avant de choisir**

Le tableau conserve les deux variantes TF-IDF et le bonus BM25. Le graphique compare le TF-IDF retenu,
BM25 et, uniquement s'il a été exécuté, l'encodeur de phrases.

Lire les **mesures calculées ci-dessous**. Aucun système n'est déclaré gagnant à l'avance.
Avec huit requêtes fictives, une seule réussite supplémentaire change Recall@1 de 0,125.
Un gain de MRR sur ce jeu ne garantit pas un gain ailleurs ; les scores bruts BM25 et les cosinus n'ont pas la même échelle.
La PCA précédente est une projection qui perd de l'information : elle n'établit pas le classement dans l'espace d'origine.


In [ ]:
import matplotlib.pyplot as plt

display(pd.DataFrame(experiments))
methods = [f'TF-IDF {NGRAM_RANGE}', 'BM25']
compared_metrics = [metrics_lexical, metrics_bm25]
if metrics_semantic is not None:
    methods.append('Embeddings')
    compared_metrics.append(metrics_semantic)
mrr_scores = [m['mrr'] for m in compared_metrics]
recall_at_3_scores = [m['recall_at_3'] for m in compared_metrics]

x = np.arange(len(methods))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, mrr_scores, width, label='MRR', color='#1f77b4', alpha=0.85, edgecolor='black')
rects2 = ax.bar(x + width/2, recall_at_3_scores, width, label='Recall@3', color='#2ca02c', alpha=0.85, edgecolor='black')

ax.set_ylabel('Scores (0.0 à 1.0)', fontsize=12)
ax.set_title('Comparaison des Performances de Recherche (Validation)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11, loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Ajout des valeurs numériques au-dessus des barres
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points de décalage vertical
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()


## **5. Décider et expliquer — puis ouvrir le test**

La correction applique un critère explicite : retenir la meilleure **MRR de validation**.
À égalité, l'ordre de préférence fixé est TF-IDF, BM25, puis embeddings. Le choix porte sur les méthodes exécutées.
On pourrait défendre un autre choix pour le coût ou la latence, mais il faut le décider avant le test.

La conclusion reprend les métriques et les prédictions réellement calculées, sans chiffres rédigés à la main.
Les erreurs listées sont des erreurs top-1 de validation ; discuter leur cause au lieu de l'inventer automatiquement.
Le test contient six requêtes différentes et n'est ouvert qu'après ce choix. Il n'est pas utilisé pour revenir sur les paramètres.

**Restitution :** deux erreurs réelles, une limite du protocole et la prochaine expérience à mener sur un corpus plus large.


In [ ]:
validation_metrics = {"tfidf": metrics_lexical, "bm25": metrics_bm25}
scorers = {"tfidf": lexical_scores, "bm25": bm25.transform}
if metrics_semantic is not None:
    validation_metrics["embedding"] = metrics_semantic
    scorers["embedding"] = semantic_scores

# Choix figé sur validation uniquement ; max conserve le premier en cas d'égalité.
CHOSEN_METHOD = max(validation_metrics, key=lambda method: validation_metrics[method]["mrr"])
print("Choix AVANT le test :", CHOSEN_METHOD, "; critère = MRR de validation")

validation_summary = " ; ".join(
    f"{name} : Recall@3={m['recall_at_3']:.3f}, MRR={m['mrr']:.3f}, n={m['n']}"
    for name, m in validation_metrics.items())
validation_errors = [row for row in analysis if row[CHOSEN_METHOD] != row["attendu"]]
error_examples = " ; ".join(
    f"{row['question']} -> {row[CHOSEN_METHOD]} (attendu : {row['attendu']})"
    for row in validation_errors[:2]) or "Aucune erreur top-1 sur cette validation minuscule."

final_scores = scorers[CHOSEN_METHOD]([q for q, _ in final_queries])
final_metrics = retrieval_metrics(final_scores, final_queries)
test_predictions = [
    {"question": q, "attendu": expected,
     "prediction": docs.id.iloc[int(final_scores[i].argmax())]}
    for i, (q, expected) in enumerate(final_queries)]
print("Test final, sans retouche après lecture :", final_metrics)
display(pd.DataFrame(test_predictions))

conclusion_binome = (
    f"VALIDATION : {validation_summary}. Choix : {CHOSEN_METHOD}, selon la MRR de validation. "
    f"ERREURS TOP-1 OBSERVÉES SUR VALIDATION : {error_examples}. "
    f"TEST RÉSERVÉ : Recall@3={final_metrics['recall_at_3']:.3f}, "
    f"MRR={final_metrics['mrr']:.3f}, n={final_metrics['n']}. "
    "LIMITES : corpus fictif de douze fiches, huit requêtes de validation et six de test. "
    "Ces résultats ne garantissent pas la qualité en production ; confirmer sur davantage de questions, "
    "dont des négations, références exactes et demandes hors périmètre.")
print(conclusion_binome)
results.update({
    "validation_tfidf": metrics_lexical, "validation_bm25": metrics_bm25,
    "validation_embedding": metrics_semantic, "experiments": experiments,
    "tfidf_ngram_range": NGRAM_RANGE, "bm25_config": {"k1": bm25.k1, "b": bm25.b},
    "embedding_model": EMBEDDING_MODEL, "embedding_executed": metrics_semantic is not None,
    "selection_metric": "mrr", "chosen_method": CHOSEN_METHOD,
    "test": final_metrics, "test_predictions": test_predictions,
    "validation_predictions": analysis, "validation_errors": validation_errors,
    "reponses": reponses, "bpe": bpe_rows})


### **Repères formateur — éléments de correction**


Attendre une décision argumentée, pas nécessairement le choix de l’encodeur. Avec six questions de test, une erreur change Recall@1 d’environ 0,167. Un cas négatif doit déclencher une discussion de l’intention. Demander pourquoi ajouter le test au corpus d’apprentissage du tokenizer ou modifier les fiches après son inspection invaliderait le protocole de comparaison initial.


## **Ressources pour prolonger**

- [HF — Tokenizers](https://huggingface.co/learn/llm-course/fr/chapter6/1)
- [Modèle multilingue et limites](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
- [scikit-learn 1.7 — TF-IDF, lissage, normalisation et tokenisation](https://scikit-learn.org/1.7/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [Stanford — Okapi BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html)
- [Lucene — variante BM25 à IDF positive](https://lucene.apache.org/core/9_9_1/core/org/apache/lucene/search/similarities/BM25Similarity.html)

Correction enrichie fournie par Chrys NIONGOLO, relue le 5 octobre 2026.
Les sorties de la version fournie ont servi à contrôler les explications ; cette édition nettoyée doit être réexécutée dans l'ordre pour produire son propre JSON.


## **Trace à rendre**

Exécuter toutes les cellules dans l'ordre puis cette dernière cellule de code.
Le JSON contient les variantes, les trois méthodes (avec `null` si les embeddings n'ont pas été exécutés),
le choix, les prédictions de validation/test et la conclusion calculée.
Si une expérience ou une réponse change, reprendre la sélection sur validation puis l'export,
sans ajuster le protocole d'après le test déjà lu. Pour une nouvelle recherche de paramètres, réserver un nouveau test.

Télécharger le JSON et le notebook avec ses sorties depuis Colab. Ne pas remplacer une mesure absente par un chiffre plausible.


In [ ]:
results["notebook"] = '01_j1_textes_recherche'
results["conclusion_binome"] = conclusion_binome
assert all(key in results for key in ["validation_bm25", "experiments", "reponses", "chosen_method", "test"])

result_file = OUTPUT / "j1_resultats.json"
result_file.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Fichier à télécharger :", result_file.resolve())
